# Build a Tool-Calling Loop


## Make an action visible before adding a framework

A user asks, “What is twelve times forty-seven, plus the shipping fee in note 3?” The proposed lab has two read-only tools: a restricted calculator and a lookup over six synthetic support notes. A model response requests a calculation, the application executes it, and the next response requests the note. A final response combines the observations. The central artifact is the complete transcript, including malformed requests and failures, so the route to an answer can be inspected rather than inferred from the final text.

This chapter implements that cycle with ordinary Python. An **agent** here is a program that lets model outputs select subsequent actions. The model produces proposals; the Python dispatcher performs actual work. **ReAct**, reasoning and acting, names an interleaving of action selection and observations. No hidden reasoning transcript is required. If a model offers a short action explanation, it can be retained as ordinary output, but correctness is established from tool requests, results, and the task checker.

The prerequisites are functions, dataclasses, JSON, exceptions, and tests. Familiarity with system, user, assistant, and tool messages helps; none of the model-training courses has to be completed first. A deterministic scripted model makes the first runs inspectable without credentials. A substantive model-generated run can then use a pinned local or hosted model adapter. The scripts show controller behavior; their success rates do not estimate a language model's ability to choose actions.

## Walk through one transcript

Start with a plain-text action protocol that exposes the parser seam. The system instruction describes two terminal shapes: an action with an allowlisted tool name and validated arguments, or a final answer. For example, `Action: calculator({"expression":"12*47"})` proposes work and `Final Answer: 570` proposes completion. The exact grammar belongs in the task fixture and parser tests. JSON arguments keep quoting rules explicit while leaving the surrounding action syntax to the learner's parser.

The application appends the raw assistant response, parses exactly one accepted outcome, validates the selected tool, and appends a typed observation. A successful calculator observation contains the result and request identifier. An error observation identifies malformed arguments or an unavailable tool without presenting fabricated data. The next model call receives the updated conversation. A final-answer proposal ends action generation, after which an independent task checker determines whether the requested computation and required lookup were satisfied.

A mixed response containing both an action and a final answer needs a documented policy. Rejecting ambiguous outputs is a reasonable default; silently selecting whichever marker appears last can execute an unintended request. Trailing commentary, embedded markers in strings, escaped quotes, empty arguments, and oversized payloads belong in parser cases. Strictness is useful only when errors become visible observations rather than unexplained crashes. A bounded corrective turn tells the model what failed and preserves the original malformed output for diagnosis.

## The interfaces the learner will implement

The proposed lesson-local package is `tool_loop`, with `types.py`, `protocol.py`, `tools.py`, `model.py`, `controller.py`, `checker.py`, and `cli.py`. A `TaskCase` stores an ID, prompt, expected fields, checking policy, and optional required tools. A `Transcript` owns ordered messages. `ModelAdapter.complete(messages, tool_specs, settings)` returns raw output, parsed-provider metadata, usage counts when available, and a request ID. It does not execute tools.

`parse_step(text)` returns one of `Action`, `FinalAnswer`, or `ParseError`. `Action` carries a name, JSON arguments, and call ID. `ToolSpec` carries a name, description, and argument schema. `ToolResult` carries `ok`, result data or an error code, and the matching call ID. `run(task, model, tools, max_steps)` returns a final proposal, terminal reason, task-checker result, and the full trace. These signatures are design targets; the files and commands are to be created in the lab, not existing Watchtower APIs.

A native tool-calling adapter is a later comparison behind the same model interface. It translates provider tool-call fields into the shared `Action` type. It must still validate arguments locally, preserve call IDs, and handle parallel calls according to an explicit policy. This lab executes one call per turn; accepting a provider's entire batch accidentally would change the step and cost accounting.

## Implement the tools without executing model text

The calculator parses an expression into a restricted syntax tree, allowing numeric literals and a bounded set of arithmetic operators. Reject names, attribute access, function calls, and arbitrary imports. Bound input length, numeric magnitude, nesting, and exponent size: even arithmetic-only input can consume unreasonable resources. Division by zero produces a typed error observation. Passing the expression to `eval`, a shell, or an unrestricted interpreter defeats the tool contract.

The note tool reads an in-memory dictionary keyed by a small integer ID. It returns the note ID, project code, fee, and text. Missing IDs are explicit failures. The model cannot provide a filesystem path, URL, or executable expression as a lookup argument. Tool dispatch occurs through an allowlisted dictionary, rather than dynamic imports or arbitrary attribute lookup. Both tools can raise internal exceptions during implementation, but the adapter converts expected failures into bounded observations and records unexpected exceptions as controlled terminal failures.

Descriptions affect whether a model chooses a tool; schemas constrain whether its request is accepted. Neither description nor valid JSON establishes authorization. This chapter's synthetic read-only inputs avoid a real permissions system. The enterprise companion course adds identity and per-resource checks without changing the distinction between a proposal and an executed operation.

## Bound the primitive loop

Even this first controller has a maximum number of model turns and a maximum observation size. A final answer ends generation; exhausting the allowance returns `step_budget` and the partial trace. Count attempted actions separately from successfully executed calls. A rejected unknown-tool proposal consumes a model turn but does not count as tool execution. A successful tool request that returns no relevant fact may still be an executed call and an unsuccessful task.

The transcript is the controller's memory. It includes the initial instructions, user task, raw proposals, results, and corrections. Keeping a complete trace is different from sending an ever-growing trace to the model; the next chapter introduces working-context selection. Persist request settings, adapter/model identity, fixture hash, and observed usage alongside messages. Temperature zero can reduce some variation, but provider behavior and local numerical execution need not be perfectly deterministic. Replayable scripted tests and measured live repeatability answer different questions.

## Build sequence and handoff

Begin by writing task cases and expected checkers, then implement and test the parser without calling a model. Implement calculator and note lookup with hostile and malformed inputs. Add a scripted adapter that returns success, parse-error recovery, unknown-tool, and wrong-final-answer sequences. Wire the controller only after these components work separately. Finally add the model adapter and an evaluation command that compares direct answering with action-based answering on the same cases.

The next chapter receives the canonical action and observation types, trace schema, and independent checkers. Its budgets and context policy wrap this small loop. A learner taking the harness chapter independently can use supplied equivalent fixtures instead of importing this implementation. A large framework, memory database, browser tool, and multi-agent delegation are optional later studies; none is needed to understand this action-observation mechanism.

## Practice and evidence

## Construct cases before scoring the controller

Prepare 15–20 versioned tasks across calculator-only, note-only, combined lookup/calculation, and no-tool questions. Include numeric formatting variants and answers whose wording can vary while the required fields stay fixed. Write exact numeric checks for arithmetic and parsed-field comparisons for notes; use substring matching only where the task definition justifies it. A correct final number obtained without a required lookup is distinguishable from a fully satisfied task. Keep those two verdicts visible.

The offline scripts include a clean calculation, malformed arguments followed by correction, unknown tool, missing note, tool exception, endless repeated action, and a plausible wrong final answer. Label each expected terminal reason and task outcome before running the controller. The fixture generator creates messages, not labels derived from the implementation. Keep transcript rendering free of credentials and real customer data.

## Proposed reproduction commands

The learner will create a lesson-local environment and implement the following command contract. These commands specify the intended artifact; no `tool_loop` package is supplied by this plan.

```bash
uv sync
uv run pytest tests -q
uv run python -m tool_loop.cli run --case calc-note-01 --model scripted --out artifacts/example.json
uv run python -m tool_loop.cli evaluate --cases fixtures/tasks.jsonl --model scripted --out artifacts/offline.jsonl
uv run python -m tool_loop.cli report --runs artifacts/offline.jsonl --out artifacts/offline-report.md
```

An opt-in model experiment adds `--model local` or `--model hosted` and writes a separate artifact directory. Record the pinned model/client versions, prompt hash, sampling settings, token allowances, and actual request metadata. For hosted calls, read secrets through the workspace vault or local runtime configuration without writing them into fixtures. Set a small spending ceiling before the run and stop when the next bounded request cannot fit the remaining allowance.

## Compare what changes and explain failures

Run direct-answer and tool-loop configurations on the same frozen task set with the same model and response settings. Report success counts, mean steps for successful tasks, failure terminal reasons, parse-error counts, attempted/executed tool calls, latency, and input/output tokens. Report no-tool cases separately to expose unnecessary work. A native-tool-calling extension may add a third configuration, allowing a comparison of parser failures while preserving the task and dispatcher.

Inspect every failure transcript. Attribute the first relevant failure to protocol parsing, tool selection, arguments, tool execution, missing evidence, answer synthesis, or checking policy. A parser error is not automatically a model-reasoning failure, and a final-answer marker is not a success label. Retain partial and failed runs in denominators. Do not publish a table of invented target scores or claim that action selection must improve every question.

Tests pin ambiguous outputs, quotes and escapes, unknown names, hostile arithmetic, numeric limits, missing notes, correction bounds, exact step boundaries, and wrong final answers. The final artifact contains the cases and checkers, a lockfile, raw sanitized transcripts, per-case results, a comparison table, and an explanation of one failure and one limitation.

All parser/tool/controller tests run on a laptop CPU at zero API cost. A small local model needs enough memory for its pinned weights and context; a hosted model needs no GPU but incurs measured usage. If resources permit only replay, complete the mechanism report and label model-selection conclusions as unmeasured. Optional extensions include native calls or a second safe tool, with the same checker and budget rather than an expanded task domain.